In [ ]:
"""
Quando selecionamos uma imagem em Markdown num Jupyter Notebook, usamos um caminho relativo ao caminho do Notebook
"""

In [1]:

from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
import librosa

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

import json
import jiwer #https://github.com/jitsi/jiwer
import time

import numpy as np

c:\Users\Admin\Desktop\ip\Automatic Speech Recognition\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<h4> Data Loader </h4>

In [2]:
with open (r"C:\Users\Admin\Desktop\ip\Automatic Speech Recognition\v2\Eval\audios.json", "r", encoding = "utf-8") as f:
    json_file = json.load (f)

display (json_file)

"""
    Normalização do Dataset.
    É importante que tanto o dataset como a transcrição obtida pelo modelo ASR tenham a mesma formatação por isso ambos vão passar por um normalização simples de remoção de duplos espaços e conversão 
    de todas as palavras para lower case.
"""

dataset = []

for exemplo in json_file:
    """
        JOIN reconstrói a frase deixando a mesma apenas com espaços em branco normais. 
        SPLIT ajuda o JOIN, separando todas as palavras da frase.
        LOWER é auto explicativo.
    """
    dataset.append (" ".join(str(exemplo["trans"]).split()).lower())

display (dataset)

[{'audio_id': 1,
  'audio_path': 'C:/Users/Admin/Desktop/ip/Automatic Speech Recognition/audio/audio1.wav',
  'trans': 'Boa tarde É aí da papelaria Sim O menina dá para me guardar dois bilhetes Dois bilhetes  Sim Que bilhetes diga-me Bilhetes lá do coiso que vai acontecer na Sexta Qual é o espetáculo diga-me É lá o que acontece lá no casino que o meu neto é que quer ir Mas eu não sei qual é Você sabe  É o da Sexta sei lá o meu neto é que me pediu isto já liguei para aqui para tantos sitíos tenho que lá ir tenho que lá ir e agora disseram-me porque é que não liga lá pá papelaria que eles guardam É assim mas você para comprar tem que vir cá pagá-los Sim está bem mas tem que mos guardar não é  Não tem de vir cá para eu tirar na máquina sair da impressora o papel para você me pagar na hora não posso guardá-los Então e depois eu chego aí ',
  'duration': 42},
 {'audio_id': 2,
  'audio_path': 'C:/Users/Admin/Desktop/ip/Automatic Speech Recognition/audio/audio2.wav',
  'trans': 'Boa tarde O m

['boa tarde é aí da papelaria sim o menina dá para me guardar dois bilhetes dois bilhetes sim que bilhetes diga-me bilhetes lá do coiso que vai acontecer na sexta qual é o espetáculo diga-me é lá o que acontece lá no casino que o meu neto é que quer ir mas eu não sei qual é você sabe é o da sexta sei lá o meu neto é que me pediu isto já liguei para aqui para tantos sitíos tenho que lá ir tenho que lá ir e agora disseram-me porque é que não liga lá pá papelaria que eles guardam é assim mas você para comprar tem que vir cá pagá-los sim está bem mas tem que mos guardar não é não tem de vir cá para eu tirar na máquina sair da impressora o papel para você me pagar na hora não posso guardá-los então e depois eu chego aí',
 'boa tarde o menina até estou nervosa que vim de lá de baixo o menina eu já encontrei encontrou o quê encontrei a elisa a então eu vou passar aqui ás relações públicas e diz está bem só um momento relações públicas boa tarde olhe o menino eu já encontrei encontrou o quê a 

<h5> Model Loader </h5>

In [3]:
MODEL_PATH = r"C:\Users\Admin\Desktop\models\ASR Models\Whisper\WhisperLv3-PT-All 4Bit"

PROCESSOR = AutoProcessor.from_pretrained (MODEL_PATH)
MODEL = AutoModelForSpeechSeq2Seq.from_pretrained (MODEL_PATH, device_map = device, dtype = torch.float16)

W0907 14:49:44.240000 8664 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights: 100%|██████████| 1259/1259 [00:01<00:00, 670.94it/s]


<hr>

<h3> Avaliação do Sistema <b> Assobio</b> Versão 1 </h3>

<p align = "center">
    <img src = "system diagram\Sistema Assobio V1.png">
</p>

In [4]:
LATENCY = []

MODEL_TRANS = []
WER = {
    "WER_CALC": []
}
CER = {
    "CER_CALC": []
}
RTF = {
    "RTF_CALC": []
}


for idx, line in enumerate (json_file):
    #print (line["audio_path"])
    WAV, SAMPLING_RATE = librosa.load (line["audio_path"], sr = 16000, mono = True)
    #print (WAV)

    """
        ###############################
    """
    torch.cuda.synchronize ()
    begin = time.time ()
    
    inputs = PROCESSOR (WAV, sampling_rate = 16000, return_tensors = "pt", truncation = False)

    inputs = inputs["input_features"].to (device, dtype = torch.float16) # Passar para GPU, não faz parte do Prefill

    with torch.inference_mode ():
        outputs = MODEL.generate (inputs, return_timestamps = True, task = "transcribe", language = "pt")

    torch.cuda.synchronize ()
    time_lat = time.time () - begin
    LATENCY.append (time_lat)

    """
        ###############################
    """

    trans = PROCESSOR.batch_decode (outputs, skip_special_tokens = True)[0]
    trans = " ".join(str(trans).split()).lower() #Normalização tal como no dataset original

    MODEL_TRANS.append (trans)

    wer = jiwer.wer (dataset[idx], trans)
    cer = jiwer.cer (dataset[idx], trans)
    rtf = time_lat / line["duration"]

    WER[line["audio_id"]] = wer
    WER["WER_CALC"].append (wer)

    CER[line["audio_id"]] = cer
    CER["CER_CALC"].append (cer)

    RTF[line["audio_id"]] = rtf
    RTF["RTF_CALC"].append (rtf)


[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
[

In [9]:
"""
Resultados
"""

print (LATENCY)
display (MODEL_TRANS)
print (WER)
print (CER)
print (RTF) #Real Time Factor -> 

print ("---" *50)

print (f"Média da Latência: {np.mean (LATENCY)} segundos")
print (f"P50 Latência: {np.percentile (LATENCY, 50)} segundos")
print (f"P95 Latência: {np.percentile (LATENCY, 95)} segundos")

print ("---" * 50)

print (f"Média Word Error Rate: {np.mean(WER['WER_CALC'])}")
print (f"Média Character Error Rate: {np.mean(CER['CER_CALC'])}")
print (f"Média Real Time Factor: {np.mean(RTF['RTF_CALC'])}")
print (f"P50 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 50)}")
print (f"P95 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 95)}")

[7.731836318969727, 9.599955558776855, 4.566570997238159, 7.220751523971558, 9.474120616912842, 7.8498406410217285, 4.175218343734741, 13.005277395248413, 4.583116769790649, 4.316499471664429, 5.332631587982178, 10.757428407669067, 6.732789754867554, 13.356438398361206]


['e foi boa tarde sim sim dois bilhotes sim bilhetes lá do do coiso que vai acontecer na sexta qual é o espetáculo me diz é lá o do que acontece lá no casino que o meu neto é que quer ir mas eu não sei qual é você sabe é o da sexta sei lá o meu neto é que me pediu já liguei para aqui para dentro do sítio tenho que lá ir tenho que lá ir e agora me porque é que não liga lá para a papelaria que eles guardam oh é assim eu mas vocêsim está bem mas tem que me os guardar não é',
 'e viu boa tardee ela acabou de la ela mora no trinta e doze seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trint

{'WER_CALC': [0.47058823529411764, 1.2342342342342343, 0.5169491525423728, 0.5882352941176471, 0.32051282051282054, 0.2937062937062937, 0.7352941176470589, 0.4682926829268293, 0.4148936170212766, 0.5, 0.6375, 0.5675675675675675, 0.5748502994011976, 0.28110599078341014], 1: 0.47058823529411764, 2: 1.2342342342342343, 3: 0.5169491525423728, 4: 0.5882352941176471, 5: 0.32051282051282054, 6: 0.2937062937062937, 7: 0.7352941176470589, 8: 0.4682926829268293, 9: 0.4148936170212766, 10: 0.5, 11: 0.6375, 12: 0.5675675675675675, 13: 0.5748502994011976, 14: 0.28110599078341014}
{'CER_CALC': [0.42718446601941745, 0.785063752276867, 0.4731369150779896, 0.5366834170854271, 0.20874471086036672, 0.23427041499330656, 0.6646464646464646, 0.3577981651376147, 0.33410672853828305, 0.4050925925925926, 0.5487364620938628, 0.4518581081081081, 0.49238578680203043, 0.19518716577540107], 1: 0.42718446601941745, 2: 0.785063752276867, 3: 0.4731369150779896, 4: 0.5366834170854271, 5: 0.20874471086036672, 6: 0.23427

<p align = "center">
    <img src = "evals\Latex Table Sistema V1.png">
</p>

<hr>

<h3> Avaliação do Sistema <b> Assobio</b> Versão 2 </h3>

<p align = "center">
    <img src = "system diagram\Sistema Assobio V2.png">
</p>

In [10]:
LATENCY = []

MODEL_TRANS = []
WER = {
    "WER_CALC": []
}
CER = {
    "CER_CALC": []
}
RTF = {
    "RTF_CALC": []
}


for idx, line in enumerate (json_file):
    #print (line["audio_path"])
    WAV, SAMPLING_RATE = librosa.load (line["audio_path"], sr = 16000, mono = True)
    #print (WAV)

    """
        ###############################
    """
    torch.cuda.synchronize ()
    begin = time.time ()
    
    inputs = PROCESSOR (WAV, sampling_rate = 16000, return_tensors = "pt", truncation = False)

    inputs = inputs["input_features"].to (device, dtype = torch.float16) # Passar para GPU, não faz parte do Prefill

    with torch.inference_mode ():
        outputs = MODEL.generate (inputs, return_timestamps = True, task = "transcribe", language = "pt", num_beams = 5)

    torch.cuda.synchronize ()
    time_lat = time.time () - begin
    LATENCY.append (time_lat)

    """
        ###############################
    """

    trans = PROCESSOR.batch_decode (outputs, skip_special_tokens = True)[0]
    trans = " ".join(str(trans).split()).lower() #Normalização tal como no dataset original

    MODEL_TRANS.append (trans)

    wer = jiwer.wer (dataset[idx], trans)
    cer = jiwer.cer (dataset[idx], trans)
    rtf = time_lat / line["duration"]

    WER[line["audio_id"]] = wer
    WER["WER_CALC"].append (wer)

    CER[line["audio_id"]] = cer
    CER["CER_CALC"].append (cer)

    RTF[line["audio_id"]] = rtf
    RTF["RTF_CALC"].append (rtf)


In [11]:
"""
Resultados
"""

print (LATENCY)
display (MODEL_TRANS)
print (WER)
print (CER)
print (RTF) #Real Time Factor -> 

print ("---" *50)

print (f"Média da Latência: {np.mean (LATENCY)} segundos")
print (f"P50 Latência: {np.percentile (LATENCY, 50)} segundos")
print (f"P95 Latência: {np.percentile (LATENCY, 95)} segundos")

print ("---" * 50)

print (f"Média Word Error Rate: {np.mean(WER['WER_CALC'])}")
print (f"Média Character Error Rate: {np.mean(CER['CER_CALC'])}")
print (f"Média Real Time Factor: {np.mean(RTF['RTF_CALC'])}")
print (f"P50 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 50)}")
print (f"P95 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 95)}")

[12.77314567565918, 31.960999488830566, 5.9332520961761475, 9.679952383041382, 12.861374378204346, 10.437676429748535, 6.554807901382446, 12.560208559036255, 6.0690062046051025, 7.483024835586548, 14.382473707199097, 16.24435329437256, 12.20352840423584, 18.88250994682312]


['e foi boa tarde sim sim dois bilhotes sim bilhetes lá do do coiso que vai acontecer na sexta qual é o espetáculo me diz é lá o do que acontece lá no casino que o meu neto é que quer ir mas eu não sei qual é você sabe é o da sexta sei lá o meu neto é que me pediu já liguei para aqui para dentro do sítio tenho que lá ir tenho que lá ir e agora disseram por que é que não liga lá para a papelaria que eles guardam oh é assim eu mas vocêcomprar tenho que vir cá los sim está bem mas tem que mas guardar não é não tenho que vir cá ponho o o isso a tirar na máquina e sair da impressora ao papel para você me pagar na hora e não posso los então e depois eu cheguei a',
 'o menino eu estou nervosa que vim do lado de baixo o menino eu já encontrei encontrei a elisa o olhe o menino eu já encontrei a elisa que ela mora aqui em baixo vocês todas as noites perguntam uma vizinha minha já me tinha dito que ela mora ali em baixo temos que ligar para ela a avisare ela é que vou denunciá la ela mora no trin

{'WER_CALC': [0.33986928104575165, 1.9189189189189189, 0.5169491525423728, 0.5637254901960784, 0.32051282051282054, 0.2937062937062937, 0.4803921568627451, 0.45365853658536587, 0.4148936170212766, 0.2619047619047619, 0.1375, 0.44594594594594594, 0.4431137724550898, 0.271889400921659], 1: 0.33986928104575165, 2: 1.9189189189189189, 3: 0.5169491525423728, 4: 0.5637254901960784, 5: 0.32051282051282054, 6: 0.2937062937062937, 7: 0.4803921568627451, 8: 0.45365853658536587, 9: 0.4148936170212766, 10: 0.2619047619047619, 11: 0.1375, 12: 0.44594594594594594, 13: 0.4431137724550898, 14: 0.271889400921659}
{'CER_CALC': [0.23578363384188628, 1.4426229508196722, 0.4731369150779896, 0.5266331658291458, 0.20874471086036672, 0.23427041499330656, 0.43434343434343436, 0.36289500509684, 0.33410672853828305, 0.1712962962962963, 0.07821901323706378, 0.31334459459459457, 0.34390862944162437, 0.16934046345811052], 1: 0.23578363384188628, 2: 1.4426229508196722, 3: 0.4731369150779896, 4: 0.5266331658291458, 5

<p align = "center">
    <img src = "evals\Latex Table Sistema V2.png">
</p>

<hr>

<h3> Avaliação do Sistema <b> Assobio</b> Versão 3 </h3>

<p align = "center">
    <img src = "system diagram\Sistema Assobio V3.png">
</p>

In [12]:
LATENCY = []

MODEL_TRANS = []
WER = {
    "WER_CALC": []
}
CER = {
    "CER_CALC": []
}
RTF = {
    "RTF_CALC": []
}


for idx, line in enumerate (json_file):
    #print (line["audio_path"])
    WAV, SAMPLING_RATE = librosa.load (line["audio_path"], sr = 16000, mono = True)
    #print (WAV)

    """
        ###############################
    """
    torch.cuda.synchronize ()
    begin = time.time ()
    
    inputs = PROCESSOR (WAV, sampling_rate = 16000, return_tensors = "pt", truncation = False)

    inputs = inputs["input_features"].to (device, dtype = torch.float16) # Passar para GPU, não faz parte do Prefill

    with torch.inference_mode ():
        outputs = MODEL.generate (inputs, return_timestamps = True, task = "transcribe", language = "pt", num_beams = 10)

    torch.cuda.synchronize ()
    time_lat = time.time () - begin
    LATENCY.append (time_lat)

    """
        ###############################
    """

    trans = PROCESSOR.batch_decode (outputs, skip_special_tokens = True)[0]
    trans = " ".join(str(trans).split()).lower() #Normalização tal como no dataset original

    MODEL_TRANS.append (trans)

    wer = jiwer.wer (dataset[idx], trans)
    cer = jiwer.cer (dataset[idx], trans)
    rtf = time_lat / line["duration"]

    WER[line["audio_id"]] = wer
    WER["WER_CALC"].append (wer)

    CER[line["audio_id"]] = cer
    CER["CER_CALC"].append (cer)

    RTF[line["audio_id"]] = rtf
    RTF["RTF_CALC"].append (rtf)


In [13]:
"""
Resultados
"""

print (LATENCY)
display (MODEL_TRANS)
print (WER)
print (CER)
print (RTF) #Real Time Factor -> 

print ("---" *50)

print (f"Média da Latência: {np.mean (LATENCY)} segundos")
print (f"P50 Latência: {np.percentile (LATENCY, 50)} segundos")
print (f"P95 Latência: {np.percentile (LATENCY, 95)} segundos")

print ("---" * 50)

print (f"Média Word Error Rate: {np.mean(WER['WER_CALC'])}")
print (f"Média Character Error Rate: {np.mean(CER['CER_CALC'])}")
print (f"Média Real Time Factor: {np.mean(RTF['RTF_CALC'])}")
print (f"P50 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 50)}")
print (f"P95 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 95)}")

[14.716593265533447, 41.859153509140015, 6.982146263122559, 11.667519330978394, 16.755046844482422, 13.636707544326782, 9.04832148551941, 16.7338445186615, 7.932849884033203, 9.364268779754639, 18.481477975845337, 23.49244737625122, 17.110599756240845, 22.890239238739014]


['e foi boa tarde sim sim dois bilhotes sim bilhetes lá do do coiso que vai acontecer na sexta qual é o espetáculo me diz é lá o do que acontece lá no casino que o meu neto é que quer ir mas eu não sei qual é você sabe é o da sexta sei lá o meu neto é que me pediu já liguei para aqui para dentro do sítio tenho que lá ir tenho que lá ir e agora disseram por que é que não liga lá para a papelaria que eles guardam oh é assim eu mas vocêcomprar tenho que vir cá e los sim está bem mas tem que mas guardar não é não tenho que vir cá e o o esse a tirar na máquina e sair da impressora ao papel para você me pagar na hora e não posso los então e depois eu cheguei a',
 'o menino eu estou nervosa que vim do lado de baixo o menino eu já encontrei encontrei a elisa o olhe o menino eu já encontrei a elisa que ela mora aqui em baixo vocês todas as noites perguntam uma vizinha minha já me tinha dito que ela mora ali em baixo temos que ligar para ela a avisare ela é que vou denunciá la ela mora no trinta

{'WER_CALC': [0.3464052287581699, 1.9189189189189189, 0.5169491525423728, 0.5637254901960784, 0.32051282051282054, 0.2937062937062937, 0.4803921568627451, 0.4585365853658537, 0.4148936170212766, 0.23809523809523808, 0.13125, 0.32882882882882886, 0.3712574850299401, 0.271889400921659], 1: 0.3464052287581699, 2: 1.9189189189189189, 3: 0.5169491525423728, 4: 0.5637254901960784, 5: 0.32051282051282054, 6: 0.2937062937062937, 7: 0.4803921568627451, 8: 0.4585365853658537, 9: 0.4148936170212766, 10: 0.23809523809523808, 11: 0.13125, 12: 0.32882882882882886, 13: 0.3712574850299401, 14: 0.271889400921659}
{'CER_CALC': [0.2302357836338419, 1.4426229508196722, 0.4731369150779896, 0.5266331658291458, 0.20874471086036672, 0.23427041499330656, 0.43434343434343436, 0.36289500509684, 0.33410672853828305, 0.16203703703703703, 0.06618531889290012, 0.19087837837837837, 0.2753807106598985, 0.16844919786096257], 1: 0.2302357836338419, 2: 1.4426229508196722, 3: 0.4731369150779896, 4: 0.5266331658291458, 5: 

<p align = "center">
    <img src = "evals\Latex Table Sistema V3.png">
</p>

<hr>

<h3> Avaliação do Sistema <b> Assobio</b> Versão 4 </h3>

<p align = "center">
    <img src = "system diagram\Sistema Assobio V4.png">
</p>

In [5]:
from silero_vad import load_silero_vad, get_speech_timestamps

VAD_MODEL = load_silero_vad ()

In [ ]:
"""
Este código calcula WER, CER, RTF e Latência do sistema Assobio com Pré Processamento de Áudio com Voice Activity Detection (VAD).
Este pedaço de código foi rodado nas 3 seguintes configurações:

    Greedy Search
    Beam Decoding = 5
    Beam Decoding = 10

Como VAD transforma um áudio em vários áudios (excertos), temos que iterar sobre os mesmos. A transcrição de um áudio é o fim do loop dos excertos do VAD e este código consegue realizar isso criando a lista ADD a 
cada áudio novo.
Também surge a questão da Latência porque a Latência corresponde ao tempo de transcrição do áudio inteiro e não apenas de um excerto, para isso usei uma variável para ir adicionando o tempo que demorava em cada excerto.
total_lat_ex = 0

Os resultados estão no MD seguinte.
"""

LATENCY = []

MODEL_TRANS = []
WER = {
    "WER_CALC": []
}
CER = {
    "CER_CALC": []
}
RTF = {
    "RTF_CALC": []
}


for idx, line in enumerate (json_file):
    #print (line["audio_path"])
    WAV, SAMPLING_RATE = librosa.load (line["audio_path"], sr = 16000, mono = True)
    #print (WAV)
    VOZ = get_speech_timestamps (WAV, VAD_MODEL)

    ADD = []
    total_lat_ex = 0

    for excertos in VOZ:

        WAV_VOZ = WAV [excertos["start"] : excertos ["end"]]

        torch.cuda.synchronize ()
        begin = time.time ()
    
        inputs = PROCESSOR (WAV_VOZ, sampling_rate = 16000, return_tensors = "pt", truncation = False) #, truncation = False
        inputs = inputs["input_features"].to (device, dtype = torch.float16) # Passar para GPU, não faz parte do Prefill

        with torch.inference_mode ():
            outputs = MODEL.generate (inputs, return_timestamps = True, task = "transcribe", language = "pt") # num_beams = 5

        torch.cuda.synchronize ()
        time_lat = time.time () - begin
        total_lat_ex += time_lat

        trans = PROCESSOR.batch_decode (outputs, skip_special_tokens = True)[0]
        trans = " ".join(str(trans).split()).lower() #Normalização tal como no dataset original

        ADD.append (trans)

    LATENCY.append (total_lat_ex)

    full_trans = " ".join (ADD)

    MODEL_TRANS.append (full_trans)

    wer = jiwer.wer (dataset[idx], full_trans)
    cer = jiwer.cer (dataset[idx], full_trans)
    rtf = total_lat_ex / line["duration"]

    WER[line["audio_id"]] = wer
    WER["WER_CALC"].append (wer)

    CER[line["audio_id"]] = cer
    CER["CER_CALC"].append (cer)

    RTF[line["audio_id"]] = rtf
    RTF["RTF_CALC"].append (rtf)


In [25]:
"""
Resultados
"""

print (LATENCY)
display (MODEL_TRANS)
print (WER)
print (CER)
print (RTF) #Real Time Factor -> 

print ("---" *50)

print (f"Média da Latência: {np.mean (LATENCY)} segundos")
print (f"P50 Latência: {np.percentile (LATENCY, 50)} segundos")
print (f"P95 Latência: {np.percentile (LATENCY, 95)} segundos")

print ("---" * 50)

print (f"Média Word Error Rate: {np.mean(WER['WER_CALC'])}")
print (f"Média Character Error Rate: {np.mean(CER['CER_CALC'])}")
print (f"Média Real Time Factor: {np.mean(RTF['RTF_CALC'])}")
print (f"P50 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 50)}")
print (f"P95 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 95)}")

[14.93299674987793, 13.822268962860107, 13.950958251953125, 47.22826385498047, 46.85388112068176, 14.717086553573608, 9.349087953567505, 24.522841930389404, 9.310574769973755, 44.15479254722595, 17.534070014953613, 34.842737674713135, 17.445170640945435, 23.648351907730103]


['é a da papelaria ó menina dá para me guardar dois bilhetes sim bilhete de lá do do coiso que vai acontecer na sexta é logo o que acontece lá no casino que o meu neto é que quer ir você sabe eu às seis dei sei lá o meu neto é que me pediu já liguei para aqui para aquele sítio tenho que ir lá ir tenho que ir lá ir e agora me disseram porque é que não liga lá para a papelaria que eles guardam mas você para comprar tem que vir cá para losnão tenho que vir cá para eu tirar na máquina sair da impressora o papel para você me pagar na hora eu não posso guardar los',
 'e boa tarde homem nem a que é toda nervosa que vem do lado de baixo ó menina eu já a encontrei controla o quê encontrei a elisa então eu vou passar aqui às relações públicas e está bem geralmente relações públicas boa tarde olha menina eu já encontrei a elisa que ela mora aqui em baixo vocês todas as noites perguntam uma vizinha minha já me tinha dito que ela mora ali em baixo temos que ligar para ela para avisar e agora o meu 

{'WER_CALC': [0.39215686274509803, 0.35135135135135137, 0.16101694915254236, 0.9607843137254902, 1.0320512820512822, 0.2727272727272727, 0.4803921568627451, 0.35609756097560974, 0.2765957446808511, 3.488095238095238, 0.3875, 0.32432432432432434, 0.3712574850299401, 0.2672811059907834], 1: 0.39215686274509803, 2: 0.35135135135135137, 3: 0.16101694915254236, 4: 0.9607843137254902, 5: 1.0320512820512822, 6: 0.2727272727272727, 7: 0.4803921568627451, 8: 0.35609756097560974, 9: 0.2765957446808511, 10: 3.488095238095238, 11: 0.3875, 12: 0.32432432432432434, 13: 0.3712574850299401, 14: 0.2672811059907834}
{'CER_CALC': [0.31622746185852985, 0.17486338797814208, 0.09185441941074524, 0.9105527638190954, 0.8251057827926658, 0.25167336010709507, 0.42424242424242425, 0.24974515800203873, 0.19489559164733178, 3.2893518518518516, 0.27557160048134777, 0.20608108108108109, 0.28045685279187815, 0.18716577540106952], 1: 0.31622746185852985, 2: 0.17486338797814208, 3: 0.09185441941074524, 4: 0.91055276381

<p align = "center">
    <img src = "evals\Latex Table Sistema V4.1.png">
</p>

<p align = "center">
    <img src = "evals\Latex Table Sistema V4.2.png">
</p>

<p align = "center">
    <img src = "evals\Latex Table Sistema V4.3.png">
</p>

<hr>

<h3> Avaliação do Sistema <b> Assobio</b> Versão 5 </h3>

<p align = "center">
    <img src = "system diagram\Sistema Assobio V5.png">
</p>

In [27]:
from demucs.api import Separator
import torchaudio

SEPARATOR_DEMUCS = Separator (model = "htdemucs")

In [ ]:
"""
Nesta versão usamos a biblioteca Demucs da Meta para podermos separar o Ruído das Vozes.
A biblioteca recebe o áudio a 44.1kHz e recebe logo um tensor PyTorch por isso temos que dar load do áudio nesse sample rate e convertido para PyTorch.
Após isso temos que converter o áudio para 16khZ e em Mono por isso usamos torchaudio para converter para 16kHz e usamos a média para converter de 2 canais para 1 canal.
"""
LATENCY = []

MODEL_TRANS = []
WER = {
    "WER_CALC": []
}
CER = {
    "CER_CALC": []
}
RTF = {
    "RTF_CALC": []
}


for idx, line in enumerate (json_file):
    #print (line["audio_path"])
    WAV, SAMPLING_RATE = librosa.load (line["audio_path"], sr = 44100, mono = False) #44.1kHz | Estéreo
    WAV = torch.from_numpy (WAV)
    
    ORIGINAL, SEPARADO = SEPARATOR_DEMUCS.separate_tensor (WAV, sr = SAMPLING_RATE)

    VOZES = SEPARADO ["vocals"]
    VOZES = torchaudio.functional.resample (VOZES, orig_freq = SAMPLING_RATE, new_freq = 16000)
    VOZES = VOZES.mean (dim = 0)

    torch.cuda.synchronize ()
    begin = time.time ()
    
    inputs = PROCESSOR (VOZES, sampling_rate = 16000, return_tensors = "pt", truncation = False) #, truncation = False
    inputs = inputs["input_features"].to (device, dtype = torch.float16) # Passar para GPU, não faz parte do Prefill

    with torch.inference_mode ():
        outputs = MODEL.generate (inputs, return_timestamps = True, task = "transcribe", language = "pt", num_beams = 5) # num_beams = 5

    torch.cuda.synchronize ()
    time_lat = time.time () - begin
    LATENCY.append (time_lat)
    
    trans = PROCESSOR.batch_decode (outputs, skip_special_tokens = True)[0]
    trans = " ".join(str(trans).split()).lower() #Normalização tal como no dataset original

    MODEL_TRANS.append (trans)

    wer = jiwer.wer (dataset[idx], trans)
    cer = jiwer.cer (dataset[idx], trans)
    rtf = time_lat / line["duration"]

    WER[line["audio_id"]] = wer
    WER["WER_CALC"].append (wer)

    CER[line["audio_id"]] = cer
    CER["CER_CALC"].append (cer)

    RTF[line["audio_id"]] = rtf
    RTF["RTF_CALC"].append (rtf)


In [33]:
"""
Resultados
"""

print (LATENCY)
display (MODEL_TRANS)
print (WER)
print (CER)
print (RTF) #Real Time Factor -> 

print ("---" *50)

print (f"Média da Latência: {np.mean (LATENCY)} segundos")
print (f"P50 Latência: {np.percentile (LATENCY, 50)} segundos")
print (f"P95 Latência: {np.percentile (LATENCY, 95)} segundos")

print ("---" * 50)

print (f"Média Word Error Rate: {np.mean(WER['WER_CALC'])}")
print (f"Média Character Error Rate: {np.mean(CER['CER_CALC'])}")
print (f"Média Real Time Factor: {np.mean(RTF['RTF_CALC'])}")
print (f"P50 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 50)}")
print (f"P95 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 95)}")

[12.910921812057495, 39.938483476638794, 7.46561074256897, 11.23920488357544, 15.534658193588257, 11.649789810180664, 7.011791229248047, 14.265023469924927, 6.2725605964660645, 5.807605981826782, 11.524980068206787, 19.695514678955078, 11.74697732925415, 19.08689570426941]


['e foi boa tarde é ido à papelaria menina dá para me guardar dois bilhetes sim bilhetes lá do do coiso que vai acontecer na sexta é logo o que acontece lá no casino que o meu neto é que quer ir é o da sexta sei lá o meu neto é que me pediu já liguei para aqui para dentro do sítio tenho que lá ir tenho que lá ir e agora disseram por que é que não liga lá para a papelaria que eles guardamcomprar tenho que vir cá e los sim está bem mas tem que me os guardar não é não tenho que vir cá e lhe o o esse a tirar na máquina e sair da impressora o papel para você me pagar na hora e não posso los então e depois eu cheguei a',
 'o menino eu estou nervosa que vim do lado de baixo o menino eu já encontrei encontrei a elisa o menino eu já encontrei a elisa que ela mora aqui em baixo vocês todas as noites perguntam uma vizinha minha já me tinha dito que ela mora ali em baixo temos que ligar para ela a avisare ela é que vou denunciá la ela mora no trinta e doce bairro não mora no trinta e doze bairro n

{'WER_CALC': [0.39215686274509803, 2.945945945945946, 0.5169491525423728, 0.5686274509803921, 0.3141025641025641, 0.2937062937062937, 0.4803921568627451, 0.45365853658536587, 0.43617021276595747, 0.5, 0.35625, 0.2747747747747748, 0.40718562874251496, 0.2534562211981567], 1: 0.39215686274509803, 2: 2.945945945945946, 3: 0.5169491525423728, 4: 0.5686274509803921, 5: 0.3141025641025641, 6: 0.2937062937062937, 7: 0.4803921568627451, 8: 0.45365853658536587, 9: 0.43617021276595747, 10: 0.5, 11: 0.35625, 12: 0.2747747747747748, 13: 0.40718562874251496, 14: 0.2534562211981567}
{'CER_CALC': [0.2787794729542302, 2.1730418943533696, 0.4731369150779896, 0.5266331658291458, 0.21015514809590974, 0.23427041499330656, 0.43434343434343436, 0.36289500509684, 0.33874709976798145, 0.4050925925925926, 0.2924187725631769, 0.13175675675675674, 0.33121827411167515, 0.15240641711229946], 1: 0.2787794729542302, 2: 2.1730418943533696, 3: 0.4731369150779896, 4: 0.5266331658291458, 5: 0.21015514809590974, 6: 0.234

<p align = "center">
    <img src = "evals\Latex Table Sistema V5.png">
</p>

<hr>

<h3> Avaliação do Sistema <b> Assobio</b> Versão 6 </h3>

<p align = "center">
    <img src = "system diagram\Sistema Assobio V6.png">
</p>


In [ ]:
import noisereduce as nr #https://github.com/timsainb/noisereduce

In [9]:

LATENCY = []

MODEL_TRANS = []
WER = {
    "WER_CALC": []
}
CER = {
    "CER_CALC": []
}
RTF = {
    "RTF_CALC": []
}


for idx, line in enumerate (json_file):
    #print (line["audio_path"])
    WAV, SAMPLING_RATE = librosa.load (line["audio_path"], sr = 16000, mono = True) 
    WAV = torch.from_numpy (WAV)
    
    WAV = nr.reduce_noise (WAV, sr = SAMPLING_RATE)

    torch.cuda.synchronize ()
    begin = time.time ()
    
    inputs = PROCESSOR (WAV, sampling_rate = 16000, return_tensors = "pt", truncation = False) #, truncation = False
    inputs = inputs["input_features"].to (device, dtype = torch.float16) # Passar para GPU, não faz parte do Prefill

    with torch.inference_mode ():
        outputs = MODEL.generate (inputs, return_timestamps = True, task = "transcribe", language = "pt", num_beams = 5) # num_beams = 5

    torch.cuda.synchronize ()
    time_lat = time.time () - begin
    LATENCY.append (time_lat)
    
    trans = PROCESSOR.batch_decode (outputs, skip_special_tokens = True)[0]
    trans = " ".join(str(trans).split()).lower() #Normalização tal como no dataset original

    MODEL_TRANS.append (trans)

    wer = jiwer.wer (dataset[idx], trans)
    cer = jiwer.cer (dataset[idx], trans)
    rtf = time_lat / line["duration"]

    WER[line["audio_id"]] = wer
    WER["WER_CALC"].append (wer)

    CER[line["audio_id"]] = cer
    CER["CER_CALC"].append (cer)

    RTF[line["audio_id"]] = rtf
    RTF["RTF_CALC"].append (rtf)


In [10]:
"""
Resultados
"""

print (LATENCY)
display (MODEL_TRANS)
print (WER)
print (CER)
print (RTF) #Real Time Factor -> 

print ("---" *50)

print (f"Média da Latência: {np.mean (LATENCY)} segundos")
print (f"P50 Latência: {np.percentile (LATENCY, 50)} segundos")
print (f"P95 Latência: {np.percentile (LATENCY, 95)} segundos")

print ("---" * 50)

print (f"Média Word Error Rate: {np.mean(WER['WER_CALC'])}")
print (f"Média Character Error Rate: {np.mean(CER['CER_CALC'])}")
print (f"Média Real Time Factor: {np.mean(RTF['RTF_CALC'])}")
print (f"P50 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 50)}")
print (f"P95 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 95)}")

[11.14275598526001, 6.547314882278442, 8.50865888595581, 14.449503898620605, 9.665930032730103, 9.474739789962769, 6.654945135116577, 8.5046067237854, 7.248936891555786, 5.382787704467773, 11.899327754974365, 19.62894320487976, 14.116288185119629, 19.648364067077637]


['e foi voltaram aí da papelaria menina dá para me guardar dois bilhetes sim bilhete lá do do coiso que vai acontecer na sexta é lá o que acontece lá no casino como é nota querida sexta é sei lá meu neto como pedeu já liguei para ela eu sinto que tenho que lá ir tenho que lá ir e agora disseram porque é que não liga lá para a papelaria que eles guardamcomprar tem que vir cá pagar os sim está bem mas temos que nos guardar não é não tem que vir cá põeme um um lixo a tirar na máquina e sair da impressora ao papel para você me pagar na hora e não posso guardá los então e depois eu cheguei',
 'olá menina eu estou nervosa que vem do lado de baixo olá menina eu já encontrei encontrei a elisa olá menina eu já encontrei a elisa que ela mora aqui em baixo vocês todas as noites perguntam uma vizinha minha já me tinha dito que ela mora ali em baixo temos que ligar para ela e avisarfalei que vou denunciar ela ela mora no trinta e do',
 'boa tarde menina é o seguinte nós portanto queriamos fazer uma

{'WER_CALC': [0.47058823529411764, 0.5405405405405406, 0.635593220338983, 0.6862745098039216, 0.5512820512820513, 0.32867132867132864, 0.5392156862745098, 0.6097560975609756, 0.35106382978723405, 0.5238095238095238, 0.2625, 0.5630630630630631, 0.40119760479041916, 0.45161290322580644], 1: 0.47058823529411764, 2: 0.5405405405405406, 3: 0.635593220338983, 4: 0.6862745098039216, 5: 0.5512820512820513, 6: 0.32867132867132864, 7: 0.5392156862745098, 8: 0.6097560975609756, 9: 0.35106382978723405, 10: 0.5238095238095238, 11: 0.2625, 12: 0.5630630630630631, 13: 0.40119760479041916, 14: 0.45161290322580644}
{'CER_CALC': [0.3217753120665742, 0.4517304189435337, 0.5199306759098787, 0.5085427135678392, 0.44710860366713684, 0.2931726907630522, 0.4383838383838384, 0.5392456676860347, 0.2552204176334107, 0.4212962962962963, 0.21299638989169675, 0.39611486486486486, 0.2893401015228426, 0.31996434937611407], 1: 0.3217753120665742, 2: 0.4517304189435337, 3: 0.5199306759098787, 4: 0.5085427135678392, 5: 

<p align = "center">
    <img src = "evals\Latex Table Sistema V6.png">
</p>

<hr>
<h3> Avaliação do Sistema <b> Assobio</b> Versão 7 </h3>

<p align = "center">
    <img src = "system diagram\Sistema Assobio V7.png">
</p>

In [11]:

LATENCY = []

MODEL_TRANS = []
WER = {
    "WER_CALC": []
}
CER = {
    "CER_CALC": []
}
RTF = {
    "RTF_CALC": []
}


for idx, line in enumerate (json_file):
    #print (line["audio_path"])
    WAV, SAMPLING_RATE = librosa.load (line["audio_path"], sr = 16000, mono = True) 
    WAV = torch.from_numpy (WAV)
    
    WAV = WAV / torch.max (torch.abs (WAV))

    torch.cuda.synchronize ()
    begin = time.time ()
    
    inputs = PROCESSOR (WAV, sampling_rate = 16000, return_tensors = "pt", truncation = False) #, truncation = False
    inputs = inputs["input_features"].to (device, dtype = torch.float16) # Passar para GPU, não faz parte do Prefill

    with torch.inference_mode ():
        outputs = MODEL.generate (inputs, return_timestamps = True, task = "transcribe", language = "pt", num_beams = 5) # num_beams = 5

    torch.cuda.synchronize ()
    time_lat = time.time () - begin
    LATENCY.append (time_lat)
    
    trans = PROCESSOR.batch_decode (outputs, skip_special_tokens = True)[0]
    trans = " ".join(str(trans).split()).lower() #Normalização tal como no dataset original

    MODEL_TRANS.append (trans)

    wer = jiwer.wer (dataset[idx], trans)
    cer = jiwer.cer (dataset[idx], trans)
    rtf = time_lat / line["duration"]

    WER[line["audio_id"]] = wer
    WER["WER_CALC"].append (wer)

    CER[line["audio_id"]] = cer
    CER["CER_CALC"].append (cer)

    RTF[line["audio_id"]] = rtf
    RTF["RTF_CALC"].append (rtf)


In [12]:
"""
Resultados
"""

print (LATENCY)
display (MODEL_TRANS)
print (WER)
print (CER)
print (RTF) #Real Time Factor -> 

print ("---" *50)

print (f"Média da Latência: {np.mean (LATENCY)} segundos")
print (f"P50 Latência: {np.percentile (LATENCY, 50)} segundos")
print (f"P95 Latência: {np.percentile (LATENCY, 95)} segundos")

print ("---" * 50)

print (f"Média Word Error Rate: {np.mean(WER['WER_CALC'])}")
print (f"Média Character Error Rate: {np.mean(CER['CER_CALC'])}")
print (f"Média Real Time Factor: {np.mean(RTF['RTF_CALC'])}")
print (f"P50 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 50)}")
print (f"P95 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 95)}")

[13.574877977371216, 31.69386076927185, 5.766404628753662, 9.465857028961182, 12.26701831817627, 10.1598801612854, 6.492882251739502, 20.720475673675537, 5.984612941741943, 5.735219955444336, 9.96632719039917, 13.266279220581055, 11.349559783935547, 18.438911199569702]


['e foi boa tarde sim sim dois bilhotes sim bilhetes lá do do coiso que vai acontecer na sexta qual é o espetáculo me diz é lá o do que acontece lá no casino que o meu neto é que quer ir mas eu não sei qual é você sabe é o da sexta sei lá o meu neto é que me pediu já liguei para aqui para dentro do sítio tenho que lá ir tenho que lá ir e agora me porque é que não liga lá para a papelaria que eles guardam oh é assim eu mas vocêcomprar tenho que vir cá los sim está bem mas tem que mas guardar não é não tenho que vir cá ponho o o isso a tirar na máquina e sair da impressora ao papel para você me pagar na hora e não posso los então e depois eu cheguei a',
 'o menino eu estou nervosa que vim do lado de baixo o menino eu já encontrei encontrei a elisa o olhe o menino eu já encontrei a elisa que ela mora aqui em baixo vocês todas as noites perguntam uma vizinha minha já me tinha dito que ela mora ali em baixo temos que ligar para ela a avisare ela é que vou denunciá la ela mora no trinta e do

{'WER_CALC': [0.32679738562091504, 1.7027027027027026, 0.5169491525423728, 0.5735294117647058, 0.3141025641025641, 0.3006993006993007, 0.4803921568627451, 0.3804878048780488, 0.4148936170212766, 0.5119047619047619, 0.4, 0.49099099099099097, 0.5269461077844312, 0.2626728110599078], 1: 0.32679738562091504, 2: 1.7027027027027026, 3: 0.5169491525423728, 4: 0.5735294117647058, 5: 0.3141025641025641, 6: 0.3006993006993007, 7: 0.4803921568627451, 8: 0.3804878048780488, 9: 0.4148936170212766, 10: 0.5119047619047619, 11: 0.4, 12: 0.49099099099099097, 13: 0.5269461077844312, 14: 0.2626728110599078}
{'CER_CALC': [0.24271844660194175, 1.1712204007285973, 0.4731369150779896, 0.5296482412060302, 0.20874471086036672, 0.23427041499330656, 0.43434343434343436, 0.27522935779816515, 0.33410672853828305, 0.4236111111111111, 0.3237063778580024, 0.38175675675675674, 0.4086294416243655, 0.17379679144385027], 1: 0.24271844660194175, 2: 1.1712204007285973, 3: 0.4731369150779896, 4: 0.5296482412060302, 5: 0.208

<p align = "center">
    <img src = "evals\Latex Table Sistema V7.png">
</p>